# Figure 2 inference

This notebook runs inference on the final round and leaderboard sets.

In [1]:
import os
import glob
import json
import pickle
from collections import defaultdict

import numpy as np
import pandas as pd
import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import average_precision_score, precision_recall_curve

sns.set_theme(style='whitegrid')

# -------------------------
# Paths
# -------------------------
ROOT_LEGACY = '/project/primate_msa/egrn/tfbs_prediction'
ROOT_PAPER = '/project/primate_msa/egrn/tfbs_prediction_paper'

TEST_SETS_PK = f'{ROOT_LEGACY}/data/test_sets.pk'
LEADERBOARD_SETS_PK = f'{ROOT_LEGACY}/data/leaderboard_sets.pk'
RAW_EMBEDDINGS_CSV = f'{ROOT_LEGACY}/data/embeddings.csv'

GENERAL_MODEL_LEGACY = f'{ROOT_LEGACY}/models/general_models/general_base_model.json'
TF_TUNED_DIR_LEGACY = f'{ROOT_LEGACY}/models/tf_models'
TF_ONLY_PK = f'{ROOT_LEGACY}/models/tf_models_nogeneral.pk'
TF_TRANSFORMER_DIR_LEGACY = f'{ROOT_LEGACY}/models/tf_transformer_models'

SCRIPT24_GENERAL_DIR = f'{ROOT_PAPER}/models/script24_crisscross_std2/general_model_tf4_chr2_8'
SCRIPT24_TF_SUMMARY = f'{ROOT_PAPER}/models/script24_crisscross_std2/tf_families_fold_summary.tsv'
SCRIPT24_TF_ONLY_DIR = f'{ROOT_PAPER}/models/script24_crisscross_std2/tf_only'
SCRIPT24_TF_TUNED_DIR = f'{ROOT_PAPER}/models/script24_crisscross_std2/tf_tuned'
SCRIPT24_TF_TRANSFORMER_TUNED_DIR = f'{ROOT_PAPER}/models/script24_crisscross_std2/tf_transformer_tuned'

SCRIPT30_ROOT = f'{ROOT_PAPER}/models/script30_general_pc100_tuned'
SCRIPT30_GENERAL_DIR = f'{SCRIPT30_ROOT}/general_30pc100_tf4_chr2_8'
SCRIPT30_TF_SUMMARY = f'{SCRIPT30_ROOT}/tf_families_fold_summary.tsv'
SCRIPT30_PCA_PKL = f'{SCRIPT30_ROOT}/pca_nt_100.pkl'

TF_TUNED_TRANSFORMER_LEGACY_GENERAL_RAWEMB_DIR = f'{ROOT_PAPER}/models/tf_tuned_transformer_legacy_general_rawemb'
TF_TUNED_TRANSFORMER_GENERAL_NT_RAWEMB_DIR = f'{ROOT_PAPER}/models/tf_tuned_transformer_general_nt_rawemb'

# -------------------------
# Feature definitions
# -------------------------
EMB_COLS_1024 = [f'emb_{i}' for i in range(1024)]
PC_COLS_100 = [f'pc_{i+1}' for i in range(100)]
GEN_FEATURES_LEGACY = [
    'maxpwm', 'cons', 'crup', 'crup_mean', 'crup_delta', 'remap', 'tf_exp', 'tf_activity',
    'tfact_crupcor_coef', 'tfact_crupcor_pval', 'trap', 'atac_min', 'atac_max', 'atac_mean', 'atac_mean_mean',
    'atac_delta_min', 'atac_delta_max', 'atac_delta_mean', 'tobias_avg', 'delta_tobias_avg', 'tobias_mean_mean', 'tobias_count',
    'cot_hits_0', 'cot_hits_1', 'cot_hits_2', 'cot_hits_3',
    'cot_maxpwm_0', 'cot_maxpwm_1', 'cot_maxpwm_2', 'cot_maxpwm_3'
]
TF_TUNED_FEATURES_LEGACY = GEN_FEATURES_LEGACY + ['xgb_general']
TF_TRANSFORMER_FEATURES_LEGACY = EMB_COLS_1024 + GEN_FEATURES_LEGACY
SCRIPT30_GENERAL_FEATURES = GEN_FEATURES_LEGACY + PC_COLS_100
SCRIPT30_TF_TUNED_FEATURES = GEN_FEATURES_LEGACY + ['xgb_general_model']

# Feature layout used by script 11 / 12 models
GEN_FEATURES_NEW = [
    'maxpwm', 'cons', 'crup', 'crup_mean', 'crup_delta', 'remap', 'tf_exp', 'tf_activity',
    'tfact_crupcor_coef', 'tfact_crupcor_pval', 'trap', 'atac_min', 'atac_max', 'atac_mean', 'atac_mean_mean',
    'atac_delta_min', 'atac_delta_max', 'atac_delta_mean', 'tobias_avg', 'delta_tobias_avg', 'tobias_mean_mean', 'tobias_count',
    'cot_hits_0', 'cot_hits_1', 'cot_hits_2', 'cot_hits_3', 'cot_hits_4', 'cot_hits_5',
    'cot_maxpwm_0', 'cot_maxpwm_1', 'cot_maxpwm_2', 'cot_maxpwm_3', 'cot_maxpwm_4', 'cot_maxpwm_5'
]
TF_TUNED_TRANSFORMER_LEGACY_GENERAL_RAWEMB_FEATURES = EMB_COLS_1024 + GEN_FEATURES_NEW + ['xgb_general']
TF_TUNED_TRANSFORMER_GENERAL_NT_RAWEMB_FEATURES = EMB_COLS_1024 + GEN_FEATURES_NEW + ['xgb_general']

In [2]:
def load_booster(path):
    b = xgb.Booster()
    b.load_model(path)
    return b

def load_sklearn_or_booster(path):
    """Try sklearn wrapper first (legacy behavior), then fallback to raw Booster for xgboost-version compatibility."""
    try:
        m = xgb.XGBRegressor()
        m.load_model(path)
        return m
    except Exception:
        return load_booster(path)

def get_model_feature_names(model, fallback):
    """Return feature names stored in model when available, else fallback list."""
    try:
        if isinstance(model, xgb.Booster):
            fn = model.feature_names
        elif hasattr(model, 'get_booster'):
            fn = model.get_booster().feature_names
        else:
            fn = None
        if fn is None:
            return list(fallback)
        return list(fn)
    except Exception:
        return list(fallback)

def get_ensemble_feature_names(boosters, fallback):
    if len(boosters) == 0:
        return list(fallback)
    fn = boosters[0].feature_names
    if fn is None:
        return list(fallback)
    return list(fn)

def predict_booster(df, features, booster):
    x = df[features].to_numpy(dtype=np.float32, copy=False)
    d = xgb.DMatrix(x, feature_names=features)
    return booster.predict(d)

def predict_xgb_model(model, df, features, use_proba=False):
    """Unified prediction for Booster / sklearn wrapper models."""
    if isinstance(model, xgb.Booster):
        return predict_booster(df, features, model)

    if use_proba and hasattr(model, 'predict_proba'):
        return model.predict_proba(df[features])[:, 1]

    return model.predict(df[features])

def ensure_columns(df, columns, fill_value=0.0):
    missing = [c for c in columns if c not in df.columns]
    if missing:
        df = df.copy()
        for c in missing:
            df[c] = fill_value
    return df

def load_ensemble_models(model_dir, tf_name, first_set=('a1','a2','b1','b2'), fallback_set=('a','b'), single_tissue_set=('c1','c2')):
    models = []

    # Prefer 4-fold
    fold_paths = [os.path.join(model_dir, f'{tf_name}_{s}.json') for s in first_set]
    if all(os.path.exists(p) for p in fold_paths):
        return [load_booster(p) for p in fold_paths]

    # New single-tissue 2-fold
    fold_paths = [os.path.join(model_dir, f'{tf_name}_{s}.json') for s in single_tissue_set]
    if all(os.path.exists(p) for p in fold_paths):
        return [load_booster(p) for p in fold_paths]

    # Legacy 2-fold
    fold_paths = [os.path.join(model_dir, f'{tf_name}_{s}.json') for s in fallback_set]
    if all(os.path.exists(p) for p in fold_paths):
        return [load_booster(p) for p in fold_paths]

    return models

def ensemble_predict(df, features, boosters):
    if len(boosters) == 0:
        raise ValueError('Empty ensemble booster list')
    d = xgb.DMatrix(df[features].to_numpy(dtype=np.float32, copy=False), feature_names=features)
    preds = [b.predict(d) for b in boosters]
    return np.mean(np.vstack(preds), axis=0)

def get_pr_metrics(y_true, y_score):
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    aupr = average_precision_score(y_true, y_score)
    return recall, precision, aupr

def no_skill_rate(y):
    return float(np.mean(y))

In [3]:
# -------------------------
# Load inputs
# -------------------------
with open(TEST_SETS_PK, 'rb') as f:
    test_sets = pickle.load(f)

with open(LEADERBOARD_SETS_PK, 'rb') as f:
    leaderboard_sets = pickle.load(f)

# Optional consistency with old notebook
if 'NANOG' in test_sets:
    del test_sets['NANOG']
if 'NANOG' in leaderboard_sets:
    del leaderboard_sets['NANOG']

raw_embeddings = pd.read_csv(RAW_EMBEDDINGS_CSV)
raw_embeddings.index = raw_embeddings['enh_id'].astype(str)
drop_cols = [c for c in ['Unnamed: 0', 'chr', 'start', 'end', 'enh_id', 'coord'] if c in raw_embeddings.columns]
raw_embeddings = raw_embeddings.drop(columns=drop_cols)
raw_embeddings.columns = EMB_COLS_1024

# -------------------------
# Load models
# -------------------------
general_legacy = load_sklearn_or_booster(GENERAL_MODEL_LEGACY)

script24_general_entries = []
script24_summary_path = f'{SCRIPT24_GENERAL_DIR}/fold_summary.tsv'
if not os.path.exists(script24_summary_path):
    raise RuntimeError(f'Missing script24 general summary: {script24_summary_path}')

script24_summary = pd.read_csv(script24_summary_path, sep='\t')
for _, row in script24_summary.iterrows():
    model_path = str(row.get('model_path', ''))
    if not model_path or not os.path.exists(model_path):
        continue
    booster = load_booster(model_path)
    script24_general_entries.append({
        'model': booster,
        'features': get_model_feature_names(booster, GEN_FEATURES_NEW),
        'stats': {
            'tf_exp_mean': float(row.get('tf_exp_mean', 0.0)),
            'tf_exp_std': float(row.get('tf_exp_std', 1.0)),
            'tf_activity_mean': float(row.get('tf_activity_mean', 0.0)),
            'tf_activity_std': float(row.get('tf_activity_std', 1.0)),
        },
    })

if len(script24_general_entries) == 0:
    raise RuntimeError(f'No script24 general models found in {script24_summary_path}')

if not os.path.exists(SCRIPT30_PCA_PKL):
    raise RuntimeError(f'Missing script30 PCA model: {SCRIPT30_PCA_PKL}')
with open(SCRIPT30_PCA_PKL, 'rb') as f:
    script30_pca = pickle.load(f)

script30_general_entries = []
script30_general_summary_path = f'{SCRIPT30_GENERAL_DIR}/fold_summary.tsv'
if not os.path.exists(script30_general_summary_path):
    raise RuntimeError(f'Missing script30 general summary: {script30_general_summary_path}')
script30_general_summary = pd.read_csv(script30_general_summary_path, sep='\t')
for _, row in script30_general_summary.iterrows():
    model_path = str(row.get('model_path', ''))
    stats_path = str(row.get('stats_path', ''))
    if (not model_path) or (not stats_path) or (not os.path.exists(model_path)) or (not os.path.exists(stats_path)):
        continue
    booster = load_booster(model_path)
    with open(stats_path, 'r') as f:
        stats_obj = json.load(f)
    script30_general_entries.append({
        'model': booster,
        'features': get_model_feature_names(booster, SCRIPT30_GENERAL_FEATURES),
        'std_features': list(stats_obj.get('std_features', [])),
        'stats': dict(stats_obj.get('stats', {})),
    })

if len(script30_general_entries) == 0:
    raise RuntimeError(f'No script30 general models found in {script30_general_summary_path}')

script30_tf_summary = pd.read_csv(SCRIPT30_TF_SUMMARY, sep='\t')
script30_tf_summary = script30_tf_summary[script30_tf_summary['family'].astype(str) == 'tf_tuned_s30'].copy()
script30_tf_tuned_models = defaultdict(list)
for _, row in script30_tf_summary.iterrows():
    tf_name = str(row.get('tf', ''))
    model_path = str(row.get('model_path', ''))
    stats_path = str(row.get('stats_path', ''))
    if tf_name == '' or (not model_path) or (not stats_path) or (not os.path.exists(model_path)) or (not os.path.exists(stats_path)):
        continue
    booster = load_booster(model_path)
    with open(stats_path, 'r') as f:
        stats_obj = json.load(f)
    script30_tf_tuned_models[tf_name].append({
        'model': booster,
        'features': get_model_feature_names(booster, SCRIPT30_TF_TUNED_FEATURES),
        'std_features': list(stats_obj.get('std_features', [])),
        'stats': dict(stats_obj.get('stats', {})),
    })

script24_tf_summary = pd.read_csv(SCRIPT24_TF_SUMMARY, sep='\t')

def load_script24_family_models(family_name, model_dir, fallback_features):
    fam = script24_tf_summary[script24_tf_summary['family'].astype(str) == family_name].copy()
    out = defaultdict(list)
    for _, row in fam.iterrows():
        tf_name = str(row.get('tf', ''))
        model_path = str(row.get('model_path', ''))
        candidate_paths = [model_path, os.path.join(model_dir, os.path.basename(model_path))]
        chosen = None
        for p in candidate_paths:
            if p and os.path.exists(p):
                chosen = p
                break
        if chosen is None or tf_name == '':
            continue

        booster = load_booster(chosen)
        out[tf_name].append({
            'model': booster,
            'features': get_model_feature_names(booster, fallback_features),
            'stats': {
                'tf_exp_mean': float(row.get('tf_exp_mean', 0.0)),
                'tf_exp_std': float(row.get('tf_exp_std', 1.0)),
                'tf_activity_mean': float(row.get('tf_activity_mean', 0.0)),
                'tf_activity_std': float(row.get('tf_activity_std', 1.0)),
            },
        })
    return out

def apply_std_features(df, std_features, stats):
    out = df.copy()
    for feat in std_features:
        if feat not in out.columns:
            continue
        mean = float(stats.get(f'{feat}_mean', 0.0))
        std = float(stats.get(f'{feat}_std', 1.0))
        if std == 0:
            std = 1.0
        out[feat] = (pd.to_numeric(out[feat], errors='coerce').fillna(0.0) - mean) / std
    return out

def predict_script24_family(df, entries):
    if len(entries) == 0:
        return np.full(len(df), np.nan, dtype=np.float32)
    preds = []
    for entry in entries:
        feats = entry['features']
        cur2 = ensure_columns(df, feats, fill_value=0.0).copy()
        cur2[feats] = cur2[feats].apply(pd.to_numeric, errors='coerce').fillna(0.0)

        tf_exp_std = entry['stats']['tf_exp_std'] if entry['stats']['tf_exp_std'] != 0 else 1.0
        tf_act_std = entry['stats']['tf_activity_std'] if entry['stats']['tf_activity_std'] != 0 else 1.0
        if 'tf_exp' in cur2.columns:
            cur2['tf_exp'] = (cur2['tf_exp'] - entry['stats']['tf_exp_mean']) / tf_exp_std
        if 'tf_activity' in cur2.columns:
            cur2['tf_activity'] = (cur2['tf_activity'] - entry['stats']['tf_activity_mean']) / tf_act_std

        preds.append(predict_booster(cur2, feats, entry['model']))
    return np.mean(np.vstack(preds), axis=0)

def predict_script24_general(df):
    return predict_script24_family(df, script24_general_entries)

def predict_script30_general(df):
    base30 = ensure_columns(df, GEN_FEATURES_LEGACY, fill_value=0.0).copy()
    base30[GEN_FEATURES_LEGACY] = base30[GEN_FEATURES_LEGACY].apply(pd.to_numeric, errors='coerce').fillna(0.0)

    emb = ensure_columns(df, EMB_COLS_1024, fill_value=0.0)[EMB_COLS_1024].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    pcs_arr = script30_pca.transform(emb.to_numpy(dtype=np.float32, copy=False)).astype(np.float32)
    pcs = pd.DataFrame(pcs_arr, index=df.index, columns=PC_COLS_100)

    x = pd.concat([base30[GEN_FEATURES_LEGACY], pcs], axis=1)
    preds = []
    for entry in script30_general_entries:
        feats = entry['features']
        x2 = ensure_columns(x, feats, fill_value=0.0).copy()
        x2[feats] = x2[feats].apply(pd.to_numeric, errors='coerce').fillna(0.0)
        x2 = apply_std_features(x2, entry['std_features'], entry['stats'])
        preds.append(predict_booster(x2, feats, entry['model']))
    return np.mean(np.vstack(preds), axis=0)

def predict_script30_tf_tuned(df, tf_name, general_pred):
    entries = script30_tf_tuned_models.get(str(tf_name), [])
    if len(entries) == 0:
        return np.full(len(df), np.nan, dtype=np.float32)

    base = ensure_columns(df, GEN_FEATURES_LEGACY, fill_value=0.0).copy()
    base[GEN_FEATURES_LEGACY] = base[GEN_FEATURES_LEGACY].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    base['xgb_general_model'] = np.asarray(general_pred, dtype=np.float32)

    preds = []
    for entry in entries:
        feats = entry['features']
        x2 = ensure_columns(base, feats, fill_value=0.0).copy()
        x2[feats] = x2[feats].apply(pd.to_numeric, errors='coerce').fillna(0.0)
        x2 = apply_std_features(x2, entry['std_features'], entry['stats'])
        preds.append(predict_booster(x2, feats, entry['model']))
    return np.mean(np.vstack(preds), axis=0)

script24_tf_only_models = load_script24_family_models('tf_only', SCRIPT24_TF_ONLY_DIR, GEN_FEATURES_NEW)
script24_tf_tuned_models = load_script24_family_models('tf_tuned', SCRIPT24_TF_TUNED_DIR, GEN_FEATURES_NEW + ['xgb_general_model'])
script24_tf_transformer_tuned_models = load_script24_family_models('tf_transformer_tuned', SCRIPT24_TF_TRANSFORMER_TUNED_DIR, EMB_COLS_1024 + GEN_FEATURES_NEW + ['xgb_general_model'])

tf_tuned_models = {}
for p in glob.glob(f'{TF_TUNED_DIR_LEGACY}/*_base_model.json'):
    tf_name = os.path.basename(p).split('_')[0]
    tf_tuned_models[tf_name] = load_sklearn_or_booster(p)
tf_tuned_models.pop('general', None)

with open(TF_ONLY_PK, 'rb') as f:
    tf_only_models = pickle.load(f)

model_order = [
    'general',
    'general_script24',
    'tf_tuned',
    'tf_only',
    'tf_transformer',
    'tf_only_script24',
    'tf_tuned_script24',
    'tf_transformer_tuned_script24',
    'general_script30_pc100',
    'tf_tuned_script30_30plus1',
    'tf_transformer_legacy_general_rawemb',
    'tf_transformer_general_nt_rawemb',
]


def evaluate_split(split_name, split_sets):
    rows = []
    pr_cache = {}

    available_tfs = sorted(set(split_sets.keys()) & set(tf_tuned_models.keys()) & set(tf_only_models.keys()))

    for tf in available_tfs:
        legacy_ens = load_ensemble_models(TF_TRANSFORMER_DIR_LEGACY, tf)
        rawemb_ens = load_ensemble_models(TF_TUNED_TRANSFORMER_LEGACY_GENERAL_RAWEMB_DIR, tf)
        general_nt_rawemb_ens = load_ensemble_models(TF_TUNED_TRANSFORMER_GENERAL_NT_RAWEMB_DIR, tf)
        if len(legacy_ens) == 0 or len(rawemb_ens) == 0 or len(general_nt_rawemb_ens) == 0:
            continue

        tf_df = split_sets[tf]
        if 'tissue' not in tf_df.columns:
            continue

        for tissue in sorted(tf_df['tissue'].dropna().astype(str).unique()):
            cur = tf_df[tf_df['tissue'].astype(str) == tissue].copy()
            if len(cur) == 0:
                continue

            cur.index = cur.index.astype(str)

            # Binary labels
            y_true = (cur['label'].astype(float).values > 2).astype(np.int8)

            # Merge raw embeddings only
            cur = cur.merge(raw_embeddings, left_index=True, right_index=True, how='left')

            # Ensure required columns exist
            cur = ensure_columns(cur, GEN_FEATURES_LEGACY, fill_value=0.0)
            cur = ensure_columns(cur, GEN_FEATURES_NEW, fill_value=0.0)
            cur = ensure_columns(cur, EMB_COLS_1024, fill_value=0.0)

            for c in set(GEN_FEATURES_LEGACY + GEN_FEATURES_NEW + EMB_COLS_1024):
                cur[c] = cur[c].astype(np.float32)

            # 1) legacy general
            general_legacy_features = get_model_feature_names(general_legacy, GEN_FEATURES_LEGACY)
            cur = ensure_columns(cur, general_legacy_features, fill_value=0.0)
            pred_general = predict_xgb_model(general_legacy, cur, general_legacy_features, use_proba=False)

            # 1b) script24 general
            pred_general_script24 = predict_script24_general(cur)
            cur['xgb_general_model'] = pred_general_script24.astype(np.float32)

            # 1c) script30 general + TF-tuned 30+1
            pred_general_script30_pc100 = predict_script30_general(cur)
            pred_tf_tuned_script30_30plus1 = predict_script30_tf_tuned(cur, tf, pred_general_script30_pc100)

            cur['xgb_general'] = pred_general.astype(np.float32)

            # 2) legacy tf_tuned
            tf_tuned_model = tf_tuned_models[tf]
            tf_tuned_features = get_model_feature_names(tf_tuned_model, TF_TUNED_FEATURES_LEGACY)
            cur = ensure_columns(cur, tf_tuned_features, fill_value=0.0)
            pred_tf_tuned = predict_xgb_model(tf_tuned_model, cur, tf_tuned_features, use_proba=False)

            # 3) legacy tf_only
            tf_only_model = tf_only_models[tf]
            tf_only_features = get_model_feature_names(tf_only_model, GEN_FEATURES_LEGACY)
            cur = ensure_columns(cur, tf_only_features, fill_value=0.0)
            if hasattr(tf_only_model, 'predict_proba'):
                pred_tf_only = tf_only_model.predict_proba(cur[tf_only_features])[:, 1]
            else:
                pred_tf_only = tf_only_model.predict(cur[tf_only_features])

            # 4) legacy transformer ensemble
            legacy_tr_features = get_ensemble_feature_names(legacy_ens, TF_TRANSFORMER_FEATURES_LEGACY)
            cur = ensure_columns(cur, legacy_tr_features, fill_value=0.0)
            pred_tf_transformer = ensemble_predict(cur, legacy_tr_features, legacy_ens)

            # 4b) script24 families
            pred_tf_only_script24 = predict_script24_family(cur, script24_tf_only_models.get(tf, []))
            pred_tf_tuned_script24 = predict_script24_family(cur, script24_tf_tuned_models.get(tf, []))
            pred_tf_transformer_tuned_script24 = predict_script24_family(cur, script24_tf_transformer_tuned_models.get(tf, []))

            # 5) script 11 ensemble
            rawemb_tr_features = get_ensemble_feature_names(rawemb_ens, TF_TUNED_TRANSFORMER_LEGACY_GENERAL_RAWEMB_FEATURES)
            cur = ensure_columns(cur, rawemb_tr_features, fill_value=0.0)
            pred_tf_transformer_legacy_general_rawemb = ensemble_predict(cur, rawemb_tr_features, rawemb_ens)

            # 6) script 12 ensemble
            general_nt_rawemb_tr_features = get_ensemble_feature_names(general_nt_rawemb_ens, TF_TUNED_TRANSFORMER_GENERAL_NT_RAWEMB_FEATURES)
            cur = ensure_columns(cur, general_nt_rawemb_tr_features, fill_value=0.0)
            pred_tf_transformer_general_nt_rawemb = ensemble_predict(cur, general_nt_rawemb_tr_features, general_nt_rawemb_ens)

            preds = {
                'general': pred_general,
                'general_script24': pred_general_script24,
                'tf_tuned': pred_tf_tuned,
                'tf_only': pred_tf_only,
                'tf_transformer': pred_tf_transformer,
                'tf_only_script24': pred_tf_only_script24,
                'tf_tuned_script24': pred_tf_tuned_script24,
                'tf_transformer_tuned_script24': pred_tf_transformer_tuned_script24,
                'general_script30_pc100': pred_general_script30_pc100,
                'tf_tuned_script30_30plus1': pred_tf_tuned_script30_30plus1,
                'tf_transformer_legacy_general_rawemb': pred_tf_transformer_legacy_general_rawemb,
                'tf_transformer_general_nt_rawemb': pred_tf_transformer_general_nt_rawemb,
            }

            pair_key = f'{tf}-{tissue}'
            pair_key_with_split = f'{split_name}:{pair_key}'
            pr_cache[pair_key_with_split] = {}

            for model_name in model_order:
                recall, precision, aupr = get_pr_metrics(y_true, preds[model_name])
                pr_cache[pair_key_with_split][model_name] = {
                    'recall': recall,
                    'precision': precision,
                    'aupr': aupr,
                }
                rows.append({
                    'split': split_name,
                    'tf': tf,
                    'tissue': tissue,
                    'tf_tissue': pair_key,
                    'tf_tissue_split': pair_key_with_split,
                    'model': model_name,
                    'aupr': aupr,
                    'no_skill': no_skill_rate(y_true),
                    'n_rows': len(y_true),
                })

    metrics_long = pd.DataFrame(rows)
    if len(metrics_long) == 0:
        metrics_wide = pd.DataFrame()
    else:
        metrics_wide = metrics_long.pivot_table(index='model', columns='tf_tissue_split', values='aupr', aggfunc='mean')

    return metrics_long, metrics_wide, pr_cache


metrics_long_test, metrics_wide_test, pr_cache_test = evaluate_split('test', test_sets)
metrics_long_lb, metrics_wide_lb, pr_cache_lb = evaluate_split('leaderboard', leaderboard_sets)

metrics_long_all = pd.concat([metrics_long_test, metrics_long_lb], axis=0, ignore_index=True)
if len(metrics_long_all) > 0:
    metrics_wide_all = metrics_long_all.pivot_table(index='model', columns='tf_tissue_split', values='aupr', aggfunc='mean')
else:
    metrics_wide_all = pd.DataFrame()

# Backward-compatible names used by figure cells below (defaults to test split)
metrics_long = metrics_long_test
metrics_wide = metrics_wide_test
pr_cache = {k.replace('test:', ''): v for k, v in pr_cache_test.items()}

print('=== Evaluation coverage ===')
print('Test TF-tissue pairs:', metrics_long_test['tf_tissue_split'].nunique() if len(metrics_long_test) else 0)
print('Leaderboard TF-tissue pairs:', metrics_long_lb['tf_tissue_split'].nunique() if len(metrics_long_lb) else 0)
print('Combined TF-tissue pairs:', metrics_long_all['tf_tissue_split'].nunique() if len(metrics_long_all) else 0)

if len(metrics_long_test):
    print('\nMean AUPR (test):')
    display(metrics_long_test.groupby('model')['aupr'].mean().sort_values(ascending=False).to_frame('mean_aupr'))
if len(metrics_long_lb):
    print('\nMean AUPR (leaderboard):')
    display(metrics_long_lb.groupby('model')['aupr'].mean().sort_values(ascending=False).to_frame('mean_aupr'))
if len(metrics_long_all):
    print('\nMean AUPR (combined):')
    display(metrics_long_all.groupby('model')['aupr'].mean().sort_values(ascending=False).to_frame('mean_aupr'))

/project/primate_msa/.venv-1/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.6.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/scratch/local/ipykernel_20894/2239411542.py:210: UserWarning: [11:36:41] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  tf_only_models = pickle.load(f)
/scratch/

=== Evaluation coverage ===
Test TF-tissue pairs: 12
Leaderboard TF-tissue pairs: 24
Combined TF-tissue pairs: 36

Mean AUPR (test):


,mean_aupr
model,
tf_transformer_tuned_script24,0.506461
tf_transformer,0.505479
tf_transformer_legacy_general_rawemb,0.503621
tf_transformer_general_nt_rawemb,0.499568
tf_tuned_script30_30plus1,0.498227
tf_tuned,0.494713
tf_tuned_script24,0.492221
tf_only_script24,0.491252
tf_only,0.476559



Mean AUPR (leaderboard):


,mean_aupr
model,
tf_transformer,0.433508
tf_transformer_legacy_general_rawemb,0.430162
tf_transformer_tuned_script24,0.428135
tf_only_script24,0.428055
tf_tuned_script30_30plus1,0.426792
tf_tuned_script24,0.424600
tf_transformer_general_nt_rawemb,0.414841
tf_tuned,0.404437
tf_only,0.401455



Mean AUPR (combined):


,mean_aupr
model,
tf_transformer,0.457498
tf_transformer_legacy_general_rawemb,0.454649
tf_transformer_tuned_script24,0.454244
tf_tuned_script30_30plus1,0.450604
tf_only_script24,0.449121
tf_tuned_script24,0.447141
tf_transformer_general_nt_rawemb,0.443083
tf_tuned,0.434529
tf_only,0.426490


In [4]:
# Export per-enhancer predictions (added export only; does not modify existing outputs)
EXPORT_PKL = f'{ROOT_PAPER}/figure_reproduction/paper_figures_3_prediction_export.pkl'

def export_prediction_tables(split_name, split_sets):
    pair_tables = {}
    rows = []

    available_tfs = sorted(set(split_sets.keys()) & set(tf_tuned_models.keys()) & set(tf_only_models.keys()))

    for tf in available_tfs:
        legacy_ens = load_ensemble_models(TF_TRANSFORMER_DIR_LEGACY, tf)
        rawemb_ens = load_ensemble_models(TF_TUNED_TRANSFORMER_LEGACY_GENERAL_RAWEMB_DIR, tf)
        general_nt_rawemb_ens = load_ensemble_models(TF_TUNED_TRANSFORMER_GENERAL_NT_RAWEMB_DIR, tf)
        if len(legacy_ens) == 0 or len(rawemb_ens) == 0 or len(general_nt_rawemb_ens) == 0:
            continue

        tf_df = split_sets[tf]
        if 'tissue' not in tf_df.columns:
            continue

        for tissue in sorted(tf_df['tissue'].dropna().astype(str).unique()):
            cur = tf_df[tf_df['tissue'].astype(str) == tissue].copy()
            if len(cur) == 0:
                continue

            cur.index = cur.index.astype(str)
            y_true = (cur['label'].astype(float).values > 2).astype(np.int8)

            cur = cur.merge(raw_embeddings, left_index=True, right_index=True, how='left')
            cur = ensure_columns(cur, GEN_FEATURES_LEGACY, fill_value=0.0)
            cur = ensure_columns(cur, GEN_FEATURES_NEW, fill_value=0.0)
            cur = ensure_columns(cur, EMB_COLS_1024, fill_value=0.0)
            for c in set(GEN_FEATURES_LEGACY + GEN_FEATURES_NEW + EMB_COLS_1024):
                cur[c] = cur[c].astype(np.float32)

            general_legacy_features = get_model_feature_names(general_legacy, GEN_FEATURES_LEGACY)
            cur = ensure_columns(cur, general_legacy_features, fill_value=0.0)
            pred_general = predict_xgb_model(general_legacy, cur, general_legacy_features, use_proba=False)

            pred_general_script24 = predict_script24_general(cur)
            cur['xgb_general_model'] = pred_general_script24.astype(np.float32)

            pred_general_script30_pc100 = predict_script30_general(cur)
            pred_tf_tuned_script30_30plus1 = predict_script30_tf_tuned(cur, tf, pred_general_script30_pc100)

            cur['xgb_general'] = pred_general.astype(np.float32)

            tf_tuned_model = tf_tuned_models[tf]
            tf_tuned_features = get_model_feature_names(tf_tuned_model, TF_TUNED_FEATURES_LEGACY)
            cur = ensure_columns(cur, tf_tuned_features, fill_value=0.0)
            pred_tf_tuned = predict_xgb_model(tf_tuned_model, cur, tf_tuned_features, use_proba=False)

            tf_only_model = tf_only_models[tf]
            tf_only_features = get_model_feature_names(tf_only_model, GEN_FEATURES_LEGACY)
            cur = ensure_columns(cur, tf_only_features, fill_value=0.0)
            if hasattr(tf_only_model, 'predict_proba'):
                pred_tf_only = tf_only_model.predict_proba(cur[tf_only_features])[:, 1]
            else:
                pred_tf_only = tf_only_model.predict(cur[tf_only_features])

            legacy_tr_features = get_ensemble_feature_names(legacy_ens, TF_TRANSFORMER_FEATURES_LEGACY)
            cur = ensure_columns(cur, legacy_tr_features, fill_value=0.0)
            pred_tf_transformer = ensemble_predict(cur, legacy_tr_features, legacy_ens)

            pred_tf_only_script24 = predict_script24_family(cur, script24_tf_only_models.get(tf, []))
            pred_tf_tuned_script24 = predict_script24_family(cur, script24_tf_tuned_models.get(tf, []))
            pred_tf_transformer_tuned_script24 = predict_script24_family(cur, script24_tf_transformer_tuned_models.get(tf, []))

            rawemb_tr_features = get_ensemble_feature_names(rawemb_ens, TF_TUNED_TRANSFORMER_LEGACY_GENERAL_RAWEMB_FEATURES)
            cur = ensure_columns(cur, rawemb_tr_features, fill_value=0.0)
            pred_tf_transformer_legacy_general_rawemb = ensemble_predict(cur, rawemb_tr_features, rawemb_ens)

            general_nt_rawemb_tr_features = get_ensemble_feature_names(general_nt_rawemb_ens, TF_TUNED_TRANSFORMER_GENERAL_NT_RAWEMB_FEATURES)
            cur = ensure_columns(cur, general_nt_rawemb_tr_features, fill_value=0.0)
            pred_tf_transformer_general_nt_rawemb = ensemble_predict(cur, general_nt_rawemb_tr_features, general_nt_rawemb_ens)

            pair_id = f'{split_name}:{tf}-{tissue}'
            out = pd.DataFrame(index=cur.index)
            out['enh_id'] = cur.index.astype(str)
            out['split'] = split_name
            out['tf'] = tf
            out['tissue'] = tissue
            out['pair_id'] = pair_id
            out['label_bin'] = y_true.astype(np.int8)
            out['motif_scan'] = pd.to_numeric(cur.get('trap', np.nan), errors='coerce')
            out['atac_seq'] = pd.to_numeric(cur.get('atac_mean', np.nan), errors='coerce')
            out[GEN_FEATURES_LEGACY] = cur[GEN_FEATURES_LEGACY].copy()

            out['pred_general'] = np.asarray(pred_general, dtype=np.float32)
            out['pred_general_script24'] = np.asarray(pred_general_script24, dtype=np.float32)
            out['pred_tf_tuned'] = np.asarray(pred_tf_tuned, dtype=np.float32)
            out['pred_tf_only'] = np.asarray(pred_tf_only, dtype=np.float32)
            out['pred_tf_transformer'] = np.asarray(pred_tf_transformer, dtype=np.float32)
            out['pred_tf_only_script24'] = np.asarray(pred_tf_only_script24, dtype=np.float32)
            out['pred_tf_tuned_script24'] = np.asarray(pred_tf_tuned_script24, dtype=np.float32)
            out['pred_tf_transformer_tuned_script24'] = np.asarray(pred_tf_transformer_tuned_script24, dtype=np.float32)
            out['pred_general_script30_pc100'] = np.asarray(pred_general_script30_pc100, dtype=np.float32)
            out['pred_tf_tuned_script30_30plus1'] = np.asarray(pred_tf_tuned_script30_30plus1, dtype=np.float32)
            out['pred_tf_transformer_legacy_general_rawemb'] = np.asarray(pred_tf_transformer_legacy_general_rawemb, dtype=np.float32)
            out['pred_tf_transformer_general_nt_rawemb'] = np.asarray(pred_tf_transformer_general_nt_rawemb, dtype=np.float32)

            pair_tables[pair_id] = out.reset_index(drop=True)
            rows.append(out.reset_index(drop=True))

    combined = pd.concat(rows, axis=0, ignore_index=True) if len(rows) else pd.DataFrame()
    return pair_tables, combined

pair_tables_test, combined_test = export_prediction_tables('test', test_sets)
pair_tables_lb, combined_lb = export_prediction_tables('leaderboard', leaderboard_sets)
pair_tables_all = {}
pair_tables_all.update(pair_tables_test)
pair_tables_all.update(pair_tables_lb)
combined_all = pd.concat([combined_test, combined_lb], axis=0, ignore_index=True) if len(combined_test) or len(combined_lb) else pd.DataFrame()

os.makedirs(os.path.dirname(EXPORT_PKL), exist_ok=True)
with open(EXPORT_PKL, 'wb') as f:
    pickle.dump({
        'pair_tables_test': pair_tables_test,
        'pair_tables_leaderboard': pair_tables_lb,
        'pair_tables_all': pair_tables_all,
        'combined_test': combined_test,
        'combined_leaderboard': combined_lb,
        'combined_all': combined_all,
        'model_order': model_order,
        'features_30': GEN_FEATURES_LEGACY,
    }, f)

print('Saved prediction export:', EXPORT_PKL)
print('Pairs test:', len(pair_tables_test))
print('Pairs leaderboard:', len(pair_tables_lb))
print('Pairs all:', len(pair_tables_all))
print('Rows all:', len(combined_all))

/scratch/local/ipykernel_20894/2128441850.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cur['xgb_general_model'] = pred_general_script24.astype(np.float32)
/scratch/local/ipykernel_20894/2128441850.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cur['xgb_general'] = pred_general.astype(np.float32)
/scratch/local/ipykernel_20894/2128441850.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all column

Saved prediction export: /project/primate_msa/egrn/tfbs_prediction_paper/figure_reproduction/paper_figures_3_prediction_export.pkl
Pairs test: 12
Pairs leaderboard: 24
Pairs all: 36
Rows all: 6499116
